# 04 — Tienda simulada en SQLite

La tienda mantiene precios, stock, carrito, cotizaciones y pedidos en SQLite. Los datos se generan con una semilla fija y la confirmación revalida stock y presupuesto dentro de una transacción atómica e idempotente.


In [1]:
%pip -q install -r requirements-lab4-modular.txt

ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements-lab4-modular.txt'
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src" / "lab4_asistente").is_dir():
    ROOT = next((p for p in ROOT.parents if (p / "src" / "lab4_asistente").is_dir()), ROOT)
sys.path.insert(0, str(ROOT / "src"))
print("Raíz del proyecto:", ROOT)
import pandas as pd
from lab4_asistente import config as C
from lab4_asistente.tienda import Tienda


Raíz del proyecto: /home/winzcode/proyectos/ADEEP_LEARNING/asistente-compras-por-voz


## Crear y restaurar la tienda

In [3]:
catalogo = pd.read_parquet(C.RUTA_DATOS / "catalogo.parquet")
C.RUTA_TIENDA.mkdir(parents=True, exist_ok=True)
BASE = C.RUTA_TIENDA / "tienda_base.sqlite"
TRABAJO = C.RUTA_TIENDA / "tienda.sqlite"

Tienda.crear_base(BASE, catalogo.product_id, C.SEMILLA, *C.PRECIO_LOGNORMAL,
                  C.PRECIO_RANGO, C.PROB_AGOTADO, C.STOCK_MAX)
tienda = Tienda(BASE, TRABAJO, C.CANTIDAD_MAX)
tienda.restaurar()
print("Huella inicial:", tienda.huella())
print("Productos:", len(catalogo))


Huella inicial: afdd683c93168902
Productos: 10688


## Probar carrito, cotización y confirmación

In [4]:
# Escoge automáticamente un producto con stock para que la demostración sea ejecutable.
producto_id = next(p for p in catalogo.product_id if tienda.producto(p)["stock"] > 0)
carrito_id = tienda.nuevo_carrito()
carrito_demo = tienda.fijar_cantidad(carrito_id, producto_id, 1)
cot = tienda.cotizar(carrito_id, presupuesto_cent=100_000)
print("Cotización:", {k: cot[k] for k in ("cotizacion_id", "total_cent", "valida", "problemas")})
if cot["valida"]:
    pedido = tienda.confirmar(cot["cotizacion_id"], presupuesto_cent=100_000)
    repetido = tienda.confirmar(cot["cotizacion_id"], presupuesto_cent=100_000)
    print("Primera confirmación:", pedido["estado"])
    print("Repetición idempotente:", repetido["estado"], "pedido", repetido.get("pedido_id"))
print("Estado final:", tienda.resumen()["stock_negativo"], "stocks negativos")


Cotización: {'cotizacion_id': 'COT-1-1', 'total_cent': 1099, 'valida': True, 'problemas': []}
Primera confirmación: registrado
Repetición idempotente: ya_registrado pedido 1
Estado final: 0 stocks negativos


### Cómo se evita una compra parcial

La tienda bloquea la escritura, revalida las condiciones, actualiza stock y registra el pedido dentro de una transacción. Si una comprobación falla, hace rollback. La clave única por cotización evita duplicar pedidos al confirmar más de una vez.